In [22]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/raw/day_approach_maskedID_timeseries.csv')
df = df.replace(-0.01, np.nan)

feature_cols = ['total km', 'nr. sessions', 'km Z3-4', 'km Z5-T1-T2',
                'km sprinting', 'strength training', 'hours alternative',
                'perceived exertion', 'perceived trainingSuccess',
                'perceived recovery']
feature_stats = []
for feature in feature_cols:
    cols = [feature]
    cols += [f"{feature}.{i}" for i in range(1, 7)]
    feature_stats.append( df[cols].agg(['max', 'mean', 'std', 'count'], axis=1).add_prefix(f"{feature}_") )
X = pd.concat(feature_stats, axis=1)

print(X.shape)
print(X.describe())

(42766, 40)
       total km_max  total km_mean  total km_std  total km_count  \
count  42766.000000   42766.000000  42766.000000         42766.0   
mean      13.853192       7.041486      5.039410             7.0   
std        8.486734       5.197299      2.859534             0.0   
min        0.000000       0.000000      0.000000             7.0   
25%        9.000000       3.242857      3.454035             7.0   
50%       13.400000       6.371429      5.126634             7.0   
75%       18.200000       9.985714      6.632352             7.0   
max       57.000000      31.471429     25.975492             7.0   

       nr. sessions_max  nr. sessions_mean  nr. sessions_std  \
count      42766.000000       42766.000000      42766.000000   
mean           1.321166           0.829264          0.447148   
std            0.569865           0.355146          0.214427   
min            0.000000           0.000000          0.000000   
25%            1.000000           0.714286          0.3

In [26]:
y = df['injury']
groups = df['Athlete ID']
print(y.shape)
print(groups.shape)
X.index.equals(y.index)
X.index.equals(groups.index)
y.sum()


(42766,)
(42766,)


np.int64(583)

In [51]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

athlete_df = (df.groupby("Athlete ID", as_index=False)["injury"].sum()
                .rename(columns={"injury": "total_injuries"}))

print("Injury count distribution:")
print(athlete_df["total_injuries"].describe().round(1), "\n")

MIN_PER_BIN = 7
injured = athlete_df["total_injuries"] > 0
n_uninjured = (~injured).sum()

athlete_df["injury_bin"] = "uninjured"
athlete_df.loc[injured, "injury_bin"] = pd.qcut(
    athlete_df.loc[injured, "total_injuries"],
    q=3, labels=["low", "mid", "high"], duplicates="drop"
).astype(str)

if n_uninjured < MIN_PER_BIN:
    athlete_df.loc[~injured, "injury_bin"] = "low"
    print(f"Only {n_uninjured} uninjured athletes: merged into 'low'")

print("Athletes per bin:")
print(athlete_df["injury_bin"].value_counts())
print("\nInjury-count range per bin:")
print(athlete_df.groupby("injury_bin")["total_injuries"].agg(["min", "max"]))

assert athlete_df["injury_bin"].value_counts().min() >= 2, "a bin is too small to stratify"

# ---------- 3. Stratified athlete split ----------
dev_athletes, holdout_athletes = train_test_split(
    athlete_df, test_size=0.15,
    stratify=athlete_df["injury_bin"], random_state=42)

# ---------- 4. Is the holdout representative? ----------
is_holdout = groups.isin(holdout_athletes["Athlete ID"])

print("\nHoldout check:")
print(f"  athletes:  {len(holdout_athletes)} of {len(athlete_df)} "
      f"({len(holdout_athletes) / len(athlete_df):.1%})")
print(f"  rows:      {is_holdout.mean():.1%}")
print(f"  injuries:  {y[is_holdout].sum() / y.sum():.1%}")
print(f"  injury rate  holdout: {y[is_holdout].mean():.2%}   dev: {y[~is_holdout].mean():.2%}")
print("  bins:", holdout_athletes["injury_bin"].value_counts().to_dict())

# ---------- 5. Save ----------
Path("data/splits").mkdir(parents=True, exist_ok=True)
(holdout_athletes[["Athlete ID"]].sort_values("Athlete ID")
    .to_csv("data/splits/holdout_athlete_ids.csv", index=False))

# ---------- 6. Dev set used for all experiments ----------
X_dev, y_dev, groups_dev = X[~is_holdout], y[~is_holdout], groups[~is_holdout]
assert X_dev.index.equals(y_dev.index) and X_dev.index.equals(groups_dev.index)
print(f"\nDev set: {X_dev.shape[0]} rows, {groups_dev.nunique()} athletes, {y_dev.sum()} injuries")

Injury count distribution:
count    74.0
mean      7.9
std       8.3
min       0.0
25%       2.0
50%       5.0
75%      11.5
max      35.0
Name: total_injuries, dtype: float64 

Athletes per bin:
injury_bin
low          22
high         21
mid          20
uninjured    11
Name: count, dtype: int64

Injury-count range per bin:
            min  max
injury_bin          
high         10   35
low           1    4
mid           5    9
uninjured     0    0

Holdout check:
  athletes:  12 of 74 (16.2%)
  rows:      15.9%
  injuries:  15.8%
  injury rate  holdout: 1.35%   dev: 1.36%
  bins: {'low': 4, 'mid': 3, 'high': 3, 'uninjured': 2}

Dev set: 35973 rows, 62 athletes, 491 injuries


In [85]:


import xgboost as xgb
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, precision_recall_curve
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
auc = []
ap = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(x, rolled['injury'], rolled['Athlete ID'])):
    training_data = rolled.iloc[train_idx].copy()

    injury_labels = training_data['injury']
    training_data = training_data.drop(columns=['injury', 'Athlete ID','start','stop'])
    
    test_data = rolled.iloc[test_idx].copy()
    test_injury_labels = test_data['injury']
    test_data = test_data.drop(columns=['injury', 'Athlete ID','start','stop'])

    neg_count = np.sum(injury_labels == 0)
    pos_count = np.sum(injury_labels == 1)
    weight_ratio = neg_count / pos_count
    
    model = xgb.XGBClassifier(max_depth=2, n_estimators = 512, scale_pos_weight = weight_ratio, learning_rate=.01) 
    model.fit(training_data, injury_labels)
    injury_pred_proba = model.predict_proba(test_data)[:, 1]
    auc_score = roc_auc_score(test_injury_labels, injury_pred_proba)
    ap_score = average_precision_score(test_injury_labels, injury_pred_proba)
    auc.append(auc_score)
    ap.append(ap_score)
    print(f"Average Precision: {ap_score:.4f}")
    print(f"Test ROC AUC: {auc_score:.4f}")
    
print(f"Average Test ROC AUC: {np.mean(auc):.4f} +/- {np.std(auc, ddof=1):.4f}")
print(f"Average Test AP: {np.mean(ap):.4f} +/- {np.std(ap, ddof=1):.4f} ")
precision, recall, thresholds = precision_recall_curve(test_injury_labels, injury_pred_proba)
#RocCurveDisplay.from_estimator(model, test_data, test_injury_labels)
disp = PrecisionRecallDisplay(precision=precision, recall=recall, average_precision=ap_score)


Average Precision: 0.0156
Test ROC AUC: 0.5707
Average Precision: 0.0171
Test ROC AUC: 0.5894
Average Precision: 0.0189
Test ROC AUC: 0.6254
Average Precision: 0.0214
Test ROC AUC: 0.6132
Average Precision: 0.0256
Test ROC AUC: 0.6555
Average Test ROC AUC: 0.6109 +/- 0.0327
Average Test AP: 0.0197 +/- 0.0039 


GroupKFold prevents leakage
Stratification handles imbalance,
combine them.

groups = Athlete ID  keeps athletes intact
y = injury (row-level) balances rare event across folds
StratifiedGroupKFold combines both

In [83]:
from sklearn.model_selection import StratifiedGroupKFold
from lifelines import CoxTimeVaryingFitter
from lifelines.utils import concordance_index
import numpy as np

x = rolled.drop(columns=['Athlete ID', 'start', 'stop', 'injury'])

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True,random_state=1)

fold_c_indices = []

def risk_set_of_c_index(df):
    injuries = df[df['injury'] == 1]
    concordant, comparable = 0.0,0.0
    for idx, inj in injuries.iterrows():
        t = inj['stop']
        athlete_id = inj['Athlete ID'] #to exclude athlete from their own risk set
        risk_set = df[ (df['stop'] >= t) &
            (t > df['start']) & 
            (df['Athlete ID'] != athlete_id) &
            (~((df['injury'] == 1) & (df['stop'] == t) ) ) ]
        #compare inj vs risk_set hazards
  
        concordant += (inj['hazard'] > risk_set['hazard']).sum() + 0.5 * (inj['hazard'] == risk_set['hazard']).sum()# A is riskier than B
        comparable += len(risk_set)
    if comparable != 0.0:
        return concordant / comparable

for fold, (train_idx, test_idx) in enumerate(sgkf.split(x, rolled['injury'], rolled['Athlete ID'])):
    training_data = rolled.iloc[train_idx].copy()
    test_data = rolled.iloc[test_idx].copy()
    #print(training_data.head())
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        df=training_data,
        id_col="Athlete ID",
        event_col="injury",
        start_col="start",
        stop_col="stop",
    )
    training_data['hazard'] = ctv.predict_partial_hazard(training_data).values
    test_data['hazard'] = ctv.predict_partial_hazard(test_data).values
    train_c = risk_set_of_c_index(training_data.copy())
    test_c = risk_set_of_c_index(test_data.copy())
    fold_c_indices.append(test_c)
    print(f"Train C Index: {train_c}")
    print(f"Test C Index: {test_c}\n")
print(f"Test Average C Index: {np.mean(fold_c_indices):.4f} +/- {np.std(fold_c_indices, ddof=1):.4f}")


        
        

Train C Index: 0.6581756607524111
Test C Index: 0.5644504748982361

Train C Index: 0.657090555887627
Test C Index: 0.5621794871794872

Train C Index: 0.649598094671033
Test C Index: 0.6212319790301442

Train C Index: 0.6734845116348006
Test C Index: 0.5266579973992198

Train C Index: 0.6612867236891038
Test C Index: 0.6538952745849298

Test Average C Index: 0.5857 +/- 0.0510
